# CaTDeL-Net: Context-Adaptive Texture Deviation Modeling for Thyroid Ultrasound
**End-to-end research implementation notebook**

Implements the manuscript's main computational ideas: image harmonization and preprocessing, optional nodule-mask segmentation, a Context-Adaptive Texture Deviation Field (TDF), compact classification with severity-aware prototype distances, simulated asynchronous multi-center federated aggregation (DPAA), evaluation, robustness checks, and explainability.

**Important:** This is a runnable research scaffold, not a clinically validated diagnostic tool. If no real labeled dataset is found, the notebook creates a small synthetic dataset only to verify that the pipeline executes. Synthetic results must never be reported as medical evidence. TN3K/DDTI do not necessarily provide the benign/malignant and severity labels required by this complete task; supply a verified `manifest.csv` with labels for those tasks.

In [ ]:
# Cell 1 — Environment and reproducibility
import os, sys, math, random, hashlib, warnings, time, copy, json, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
try:
    import cv2
except ImportError:
    !pip -q install opencv-python-headless
    import cv2
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
except ImportError:
    !pip -q install torch
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, matthews_corrcoef, roc_auc_score, average_precision_score,
    brier_score_loss, log_loss, balanced_accuracy_score)
from sklearn.calibration import calibration_curve
from sklearn.utils.class_weight import compute_class_weight
from scipy.ndimage import binary_erosion, distance_transform_edt
from scipy.stats import entropy
from PIL import Image

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "| OpenCV:", cv2.__version__, "| PyTorch:", torch.__version__)

## 1. Configure data paths

Prepare a CSV manifest with at least `image_path` and `label` columns. `label` should be `benign` or `malignant` (or `0`/`1`). Optional columns: `center`, `mask_path`, `severity` (`low`, `medium`, `high`, `critical`), `shape`, `margin`, `calcification`, `composition`, `echogenicity`, and `patient_id`.

Paths may be absolute or relative to `DATA_ROOT`. For segmentation, `mask_path` should point to a binary nodule mask. Use patient-level splitting where multiple images belong to one patient.

In [ ]:
# Cell 2 — Paths and task configuration
DATA_ROOT = Path("/content/thyroid_data")  # Change this to your dataset folder
MANIFEST_PATH = DATA_ROOT / "manifest.csv"
OUTPUT_DIR = Path("/content/catdel_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_SIZE = 128
BATCH_SIZE = 16
EPOCHS = 5                 # Increase for real experiments
FEDERATED_ROUNDS = 3
LOCAL_EPOCHS = 1
NUM_CLIENTS = 4
USE_SYNTHETIC_IF_MISSING = True
print("DATA_ROOT:", DATA_ROOT, "| manifest exists:", MANIFEST_PATH.exists())

## 2. Load data and perform quality control

The loader removes exact/perceptual duplicates and resolves paths. Robust Z-score statistics are recorded by center. If a real manifest is unavailable, a generated toy dataset is used strictly as a pipeline smoke test.

In [ ]:
# Cell 3 — Manifest loader, duplicate removal, synthetic smoke-test data
def make_synthetic_dataset(n=240, size=IMAGE_SIZE):
    rng = np.random.default_rng(SEED)
    rows = []
    synth_dir = OUTPUT_DIR / "synthetic_images"
    synth_dir.mkdir(parents=True, exist_ok=True)
    for i in range(n):
        y = i % 2
        center = f"Center_{i % NUM_CLIENTS + 1}"
        img = rng.normal(0.45, 0.12, (size, size)).clip(0, 1)
        yy, xx = np.ogrid[:size, :size]
        cx, cy = size//2 + rng.integers(-12, 13), size//2 + rng.integers(-12, 13)
        rx = rng.integers(18, 32); ry = rng.integers(15, 29)
        lesion = ((xx-cx)/rx)**2 + ((yy-cy)/ry)**2 <= 1
        img[lesion] += 0.15 if y == 0 else -0.10
        if y == 1:
            for _ in range(18):
                px, py = rng.integers(0,size,2)
                img[max(0,py-1):py+2,max(0,px-1):px+2] += 0.22
        img = (img.clip(0,1)*255).astype(np.uint8)
        mask = (lesion.astype(np.uint8)*255)
        ip = synth_dir / f"image_{i:04d}.png"
        mp = synth_dir / f"mask_{i:04d}.png"
        cv2.imwrite(str(ip), img); cv2.imwrite(str(mp), mask)
        rows.append({"image_path":str(ip), "mask_path":str(mp),
                     "label": "malignant" if y else "benign",
                     "center":center, "severity":["low","medium","high","critical"][i%4],
                     "patient_id":f"P{i:04d}"})
    return pd.DataFrame(rows)

def resolve_path(p):
    p = Path(str(p))
    return p if p.is_absolute() else DATA_ROOT / p

if MANIFEST_PATH.exists():
    df = pd.read_csv(MANIFEST_PATH)
    if not {"image_path","label"}.issubset(df.columns):
        raise ValueError("manifest.csv must contain image_path and label columns")
    df["image_path"] = df["image_path"].map(resolve_path)
    if "mask_path" in df:
        df["mask_path"] = df["mask_path"].map(lambda x: resolve_path(x) if pd.notna(x) and str(x).strip() else None)
    else:
        df["mask_path"] = None
    if "center" not in df: df["center"] = "Center_1"
    if "patient_id" not in df: df["patient_id"] = [f"row_{i}" for i in range(len(df))]
    df = df[df.image_path.map(lambda p: Path(p).exists())].copy()
    if len(df) < 10:
        raise ValueError("Fewer than 10 valid image files found. Check image_path values.")
    df["label"] = df["label"].astype(str).str.lower().map(
        {"benign":0,"malignant":1,"0":0,"1":1,"negative":0,"positive":1})
    if df["label"].isna().any():
        raise ValueError("Unsupported labels. Use benign/malignant or 0/1.")
    df["label"] = df["label"].astype(int)
    # Perceptual-hash-like duplicate removal: grayscale thumbnail and robust hash.
    def image_hash(path):
        im = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if im is None: return None
        im = cv2.resize(im, (8,8), interpolation=cv2.INTER_AREA)
        return hashlib.md5((im > im.mean()).astype(np.uint8).tobytes()).hexdigest()
    df["phash"] = df.image_path.map(image_hash)
    df = df.dropna(subset=["phash"]).drop_duplicates("phash").drop(columns="phash").reset_index(drop=True)
    DATA_MODE = "REAL"
else:
    if not USE_SYNTHETIC_IF_MISSING:
        raise FileNotFoundError(f"Missing {MANIFEST_PATH}. Create the manifest or set USE_SYNTHETIC_IF_MISSING=True.")
    df = make_synthetic_dataset()
    DATA_MODE = "SYNTHETIC_SMOKE_TEST"

print("Data mode:", DATA_MODE, "| rows after QC:", len(df))
display(df.head())
print(df["label"].value_counts().rename(index={0:"benign",1:"malignant"}))

## 3. Ultrasound preprocessing

Pipeline: speckle suppression (SRAD-inspired iterative diffusion approximation), CLAHE, percentile clipping, median-IQR normalization, and ROI-preserving resize. This implementation is intentionally lightweight and dependency-friendly; the diffusion filter is an approximation rather than a verified clinical SRAD implementation.

In [ ]:
# Cell 4 — Image preprocessing and quality-control preview
def srad_inspired_diffusion(img, iterations=4, lam=0.15):
    x = img.astype(np.float32) / 255.0
    for _ in range(iterations):
        n = np.roll(x, -1, axis=0)-x
        s = np.roll(x, 1, axis=0)-x
        e = np.roll(x, -1, axis=1)-x
        w = np.roll(x, 1, axis=1)-x
        grad = (n*n+s*s+e*e+w*w)
        local = cv2.blur(x*x, (5,5)) - cv2.blur(x, (5,5))**2
        coeff = 1.0 / (1.0 + grad / (local + 1e-4))
        x = np.clip(x + lam*coeff*(n+s+e+w), 0, 1)
    return (x*255).astype(np.uint8)

def preprocess_image(path, size=IMAGE_SIZE):
    im = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if im is None: raise ValueError(f"Could not read image: {path}")
    im = srad_inspired_diffusion(im)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    im = clahe.apply(im)
    lo, hi = np.percentile(im, [1,99])
    im = np.clip(im, lo, hi)
    med = np.median(im); q1, q3 = np.percentile(im,[25,75])
    im = (im-med)/(q3-q1+1e-6)
    # Preserve aspect ratio and pad rather than stretch.
    h,w = im.shape; scale = min(size/h, size/w)
    nh,nw = max(1,int(h*scale)), max(1,int(w*scale))
    im = cv2.resize(im,(nw,nh),interpolation=cv2.INTER_AREA)
    canvas = np.zeros((size,size),dtype=np.float32)
    y0=(size-nh)//2; x0=(size-nw)//2
    canvas[y0:y0+nh,x0:x0+nw]=im
    # Robustly map to approximately [-1,1]
    canvas=np.clip(canvas,-4,4)/4
    return canvas.astype(np.float32)

sample = preprocess_image(df.iloc[0].image_path)
plt.figure(figsize=(4,4)); plt.imshow(sample,cmap="gray"); plt.title("Preprocessed ultrasound"); plt.axis("off"); plt.show()

## 4. Dataset and lightweight nodule segmentation

If binary masks are available, a compact U-Net-style segmenter can be trained. If masks are absent, segmentation training is skipped; for downstream TDF extraction a coarse central ROI is used, and segmentation metrics are marked unavailable. Do not interpret the fallback ROI as a true nodule localization result.

In [ ]:
# Cell 5 — Dataset, U-Net, and optional segmentation training
class ThyroidDataset(Dataset):
    def __init__(self, frame, with_masks=False):
        self.frame=frame.reset_index(drop=True); self.with_masks=with_masks
    def __len__(self): return len(self.frame)
    def __getitem__(self, idx):
        r=self.frame.iloc[idx]
        x=preprocess_image(r.image_path)
        y=int(r.label)
        out={"image":torch.tensor(x[None],dtype=torch.float32),
             "label":torch.tensor(y,dtype=torch.long),
             "index":torch.tensor(idx)}
        mp=r.get("mask_path",None)
        if self.with_masks and mp is not None and pd.notna(mp) and Path(str(mp)).exists():
            m=cv2.imread(str(mp),cv2.IMREAD_GRAYSCALE)
            m=cv2.resize(m,(IMAGE_SIZE,IMAGE_SIZE),interpolation=cv2.INTER_NEAREST)
            out["mask"]=torch.tensor((m>0)[None].astype(np.float32))
        return out

class ConvBlock(nn.Module):
    def __init__(self, a,b):
        super().__init__()
        self.net=nn.Sequential(nn.Conv2d(a,b,3,padding=1,bias=False),nn.BatchNorm2d(b),nn.ReLU(inplace=True),
                               nn.Conv2d(b,b,3,padding=1,bias=False),nn.BatchNorm2d(b),nn.ReLU(inplace=True))
    def forward(self,x): return self.net(x)

class LiteUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.e1=ConvBlock(1,16); self.e2=ConvBlock(16,32); self.mid=ConvBlock(32,48)
        self.pool=nn.MaxPool2d(2)
        self.d2=ConvBlock(48+32,32); self.d1=ConvBlock(32+16,16)
        self.out=nn.Conv2d(16,1,1)
    def forward(self,x):
        a=self.e1(x); b=self.e2(self.pool(a)); c=self.mid(self.pool(b))
        u=F.interpolate(c,size=b.shape[-2:],mode="bilinear",align_corners=False)
        u=self.d2(torch.cat([u,b],1))
        u=F.interpolate(u,size=a.shape[-2:],mode="bilinear",align_corners=False)
        return self.out(self.d1(torch.cat([u,a],1)))

mask_available = ("mask_path" in df.columns) and df.mask_path.map(lambda p: p is not None and pd.notna(p) and Path(str(p)).exists()).any()
segmenter = LiteUNet().to(DEVICE)
segmentation_trained = False
if mask_available:
    mdf=df[df.mask_path.map(lambda p: p is not None and pd.notna(p) and Path(str(p)).exists())].copy()
    seg_loader=DataLoader(ThyroidDataset(mdf,with_masks=True),batch_size=BATCH_SIZE,shuffle=True,num_workers=0)
    opt=torch.optim.Adam(segmenter.parameters(),lr=1e-3)
    for ep in range(2 if DATA_MODE.startswith("SYNTHETIC") else EPOCHS):
        segmenter.train(); losses=[]
        for b in seg_loader:
            if "mask" not in b: continue
            x=b["image"].to(DEVICE); y=b["mask"].to(DEVICE)
            opt.zero_grad(); logits=segmenter(x)
            loss=F.binary_cross_entropy_with_logits(logits,y)
            loss.backward(); opt.step(); losses.append(loss.item())
        print(f"Segmentation epoch {ep+1}: loss={np.mean(losses):.4f}" if losses else "No valid mask batches")
    segmentation_trained=True
else:
    print("No usable masks found: segmentation model initialized but not trained.")

## 5. CaTDeL-Net texture deviation field

The observed texture is encoded with depthwise convolution. A context predictor estimates expected texture; `TDF = observed - expected`. Four statistics are calculated (mean, variance, entropy, spatial concentration), then a compact CNN with channel attention produces the representation used for classification.

In [ ]:
# Cell 6 — CaTDeL-Net
class DepthwiseBlock(nn.Module):
    def __init__(self,c):
        super().__init__()
        self.dw=nn.Conv2d(c,c,3,padding=1,groups=c,bias=False)
        self.pw=nn.Conv2d(c,c,1,bias=False)
        self.bn=nn.BatchNorm2d(c)
    def forward(self,x): return F.relu(self.bn(self.pw(self.dw(x))))

class CaTDeLNet(nn.Module):
    def __init__(self, embed_dim=64):
        super().__init__()
        self.context=nn.Sequential(nn.Conv2d(1,16,3,padding=1,bias=False),nn.BatchNorm2d(16),nn.ReLU(),
                                   DepthwiseBlock(16),nn.Conv2d(16,1,1))
        self.expected=nn.Sequential(nn.Conv2d(1,8,3,padding=1),nn.ReLU(),nn.Conv2d(8,1,3,padding=1))
        self.backbone=nn.Sequential(nn.Conv2d(2,16,3,padding=1,stride=2),nn.BatchNorm2d(16),nn.ReLU(),
                                    DepthwiseBlock(16),nn.Conv2d(16,32,3,padding=1,stride=2),nn.BatchNorm2d(32),nn.ReLU(),
                                    DepthwiseBlock(32),nn.Conv2d(32,64,3,padding=1,stride=2),nn.ReLU(),
                                    nn.AdaptiveAvgPool2d(1))
        self.channel_gate=nn.Sequential(nn.Linear(64,16),nn.ReLU(),nn.Linear(16,64),nn.Sigmoid())
        self.project=nn.Linear(64+4,embed_dim)
        self.classifier=nn.Sequential(nn.Linear(embed_dim,32),nn.ReLU(),nn.Dropout(0.2),nn.Linear(32,2))
        self.severity=nn.Linear(embed_dim,4)
    def forward(self,x,return_aux=False):
        context=self.context(x)
        expected=self.expected(context)
        tdf=x-expected
        abs_tdf=tdf.abs()
        mean=tdf.mean((2,3))
        var=tdf.var((2,3),unbiased=False)
        # Histogram entropy per image (differentiable statistics are approximated via fixed bins).
        hist=[]
        for j in range(x.shape[0]):
            vals=abs_tdf[j].flatten()
            probs=torch.stack([((vals>=k/8)&(vals<(k+1)/8)).float().mean() for k in range(32)])
            hist.append(-(probs*(probs+1e-8).log()).sum().view(1))
        ent=torch.cat(hist).view(-1,1)
        flat=abs_tdf.flatten(2)
        topk=max(1,flat.shape[-1]//10)
        concentration=flat.topk(topk,dim=-1).values.mean(-1)/(flat.mean(-1)+1e-6)
        stats=torch.cat([mean,var,ent,concentration],dim=1)
        feat=self.backbone(torch.cat([x,tdf],dim=1)).flatten(1)
        feat=feat*self.channel_gate(feat)
        z=self.project(torch.cat([feat,stats],dim=1))
        logits=self.classifier(z); sev=self.severity(z)
        if return_aux: return logits,sev,z,tdf,stats
        return logits

## 6. Train/test split and diagnostic model

A stratified split is used by default. For repeated scans per person, replace this with a patient-group split to prevent leakage. Classifier targets: benign/malignant. Severity is an optional auxiliary target; missing severity labels are masked out during loss calculation.

In [ ]:
# Cell 7 — Train and evaluate CaTDeL-Net
# Normalize labels and optional severity labels
severity_map={"low":0,"medium":1,"high":2,"critical":3}
df["severity_id"] = df["severity"].astype(str).str.lower().map(severity_map) if "severity" in df.columns else np.nan
idx=np.arange(len(df))
train_idx,test_idx=train_test_split(idx,test_size=0.2,random_state=SEED,stratify=df.label)
train_df=df.iloc[train_idx].copy(); test_df=df.iloc[test_idx].copy()
train_loader=DataLoader(ThyroidDataset(train_df),batch_size=BATCH_SIZE,shuffle=True,num_workers=0)
test_loader=DataLoader(ThyroidDataset(test_df),batch_size=BATCH_SIZE,shuffle=False,num_workers=0)
model=CaTDeLNet().to(DEVICE)
weights=compute_class_weight(class_weight="balanced",classes=np.array([0,1]),y=train_df.label.to_numpy())
criterion=nn.CrossEntropyLoss(weight=torch.tensor(weights,dtype=torch.float32,device=DEVICE))
optimizer=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
history={"loss":[],"accuracy":[]}
epochs_run=2 if DATA_MODE.startswith("SYNTHETIC") else EPOCHS
for ep in range(epochs_run):
    model.train(); losses=[]; correct=total=0
    for b in train_loader:
        x=b["image"].to(DEVICE); y=b["label"].to(DEVICE)
        optimizer.zero_grad()
        logits,sev,z,tdf,stats=model(x,return_aux=True)
        loss=criterion(logits,y)
        # Auxiliary severity learning only where labels exist
        batch_indices=b["index"].cpu().numpy()
        sev_targets=df.iloc[train_idx[batch_indices]]["severity_id"].to_numpy()
        valid=np.isfinite(sev_targets)
        if valid.any():
            st=torch.tensor(sev_targets[valid].astype(int),device=DEVICE)
            loss=loss+0.15*F.cross_entropy(sev[torch.tensor(valid,device=DEVICE)],st)
        loss.backward(); optimizer.step()
        losses.append(loss.item()); correct+=(logits.argmax(1)==y).sum().item(); total+=len(y)
    history["loss"].append(float(np.mean(losses))); history["accuracy"].append(correct/max(total,1))
    print(f"Epoch {ep+1}/{epochs_run} | loss={history['loss'][-1]:.4f} | train accuracy={history['accuracy'][-1]:.3f}")

plt.figure(figsize=(6,4)); plt.plot(history["loss"],marker="o",label="Loss"); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.grid(alpha=.3); plt.show()

@torch.no_grad()
def predict_loader(model,loader):
    model.eval(); ys=[]; ps=[]; zs=[]; tdfl=[]
    for b in loader:
        x=b["image"].to(DEVICE)
        logits,sev,z,tdf,stats=model(x,return_aux=True)
        ys.extend(b["label"].numpy().tolist())
        ps.extend(torch.softmax(logits,1)[:,1].cpu().numpy().tolist())
        zs.extend(z.cpu().numpy()); tdfl.extend(tdf.cpu().numpy())
    return np.array(ys),np.array(ps),np.array(zs),np.array(tdfl)
y_test, p_test, z_test, tdf_test=predict_loader(model,test_loader)

## 7. Deviation-margin prototype classifier and diagnostic metrics

The classifier combines learned representation with distance to benign/malignant prototypes. The learned logits remain part of the score; prototype distances add a transparent similarity signal. Metrics are computed from held-out data.

In [ ]:
# Cell 8 — Prototype classifier and core metrics
@torch.no_grad()
def collect_embeddings(model, loader):
    model.eval(); ys=[]; zs=[]
    for b in loader:
        _,_,z,_,_=model(b["image"].to(DEVICE),return_aux=True)
        ys.extend(b["label"].numpy()); zs.extend(z.cpu().numpy())
    return np.asarray(ys),np.asarray(zs)
y_train_emb,z_train=collect_embeddings(model,train_loader)
prototypes={c:z_train[y_train_emb==c].mean(0) for c in np.unique(y_train_emb)}
dist0=np.linalg.norm(z_test-prototypes.get(0,np.zeros(z_test.shape[1])),axis=1)
dist1=np.linalg.norm(z_test-prototypes.get(1,np.zeros(z_test.shape[1])),axis=1)
proto_prob=1/(1+np.exp(np.clip(dist1-dist0,-30,30)))
p_final=np.clip(0.75*p_test+0.25*proto_prob,1e-6,1-1e-6)
y_pred=(p_final>=0.5).astype(int)

def classification_metrics(y,p,threshold=0.5):
    pred=(p>=threshold).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {
      "Accuracy":accuracy_score(y,pred),
      "Precision":precision_score(y,pred,zero_division=0),
      "Sensitivity/Recall":recall_score(y,pred,zero_division=0),
      "Specificity":tn/(tn+fp+1e-12),
      "F1":f1_score(y,pred,zero_division=0),
      "MCC":matthews_corrcoef(y,pred) if len(np.unique(pred))>1 else 0.0,
      "ROC-AUC":roc_auc_score(y,p) if len(np.unique(y))>1 else np.nan,
      "PR-AUC":average_precision_score(y,p) if len(np.unique(y))>1 else np.nan,
      "Brier Score":brier_score_loss(y,p),
      "Log Loss":log_loss(y,p,labels=[0,1])
    }
metrics_df=pd.DataFrame([classification_metrics(y_test,p_final)]).T.reset_index()
metrics_df.columns=["Metric","Value"]
display(metrics_df)
cm=confusion_matrix(y_test,y_pred,labels=[0,1])
plt.figure(figsize=(4,4)); plt.imshow(cm); plt.xticks([0,1],["Benign","Malignant"]); plt.yticks([0,1],["Benign","Malignant"])
for (i,j),v in np.ndenumerate(cm): plt.text(j,i,str(v),ha="center",va="center",fontsize=12)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Held-out confusion matrix"); plt.colorbar(); plt.show()

## 8. DPAA asynchronous federated collaborative learning

Each simulated center trains a local copy. Server aggregation weights updates by sample count, update staleness, validation confidence, and prototype consistency. This is a single-process simulation of the manuscript's DPAA mechanism, not a secure production federated-learning deployment. No raw images are sent between simulated clients in the aggregation step.

In [ ]:
# Cell 9 — DPAA-style federated simulation
def state_distance(a,b):
    vals=[]
    for k in a:
        if torch.is_floating_point(a[k]):
            vals.append(torch.mean((a[k].detach().cpu()-b[k].detach().cpu())**2).item())
    return float(np.mean(vals)) if vals else 0.0

def dpaa_aggregate(global_state, client_states, sample_counts, confidences, ages, proto_vectors=None):
    # Staleness-aware and validation-confidence-aware weights.
    raw=[]
    for n,c,age in zip(sample_counts,confidences,ages):
        raw.append(max(n,1)*max(c,0.05)/(1.0+max(age,0)))
    raw=np.asarray(raw,dtype=float)
    if proto_vectors is not None and len(proto_vectors)>1:
        pmean=np.mean(proto_vectors,axis=0)
        consistency=np.array([1/(1+np.linalg.norm(p-pmean)) for p in proto_vectors])
        raw*=consistency
    weights=raw/(raw.sum()+1e-12)
    new_state={}
    for k in global_state:
        if torch.is_floating_point(global_state[k]):
            new_state[k]=sum(float(w)*cs[k].to(global_state[k].device) for w,cs in zip(weights,client_states))
        else:
            new_state[k]=global_state[k]
    return new_state,weights

def train_local(model, loader, epochs=1):
    local=copy.deepcopy(model).to(DEVICE)
    opt=torch.optim.AdamW(local.parameters(),lr=5e-4,weight_decay=1e-4)
    local.train()
    for _ in range(epochs):
        for b in loader:
            x=b["image"].to(DEVICE); y=b["label"].to(DEVICE)
            opt.zero_grad(); loss=F.cross_entropy(local(x),y); loss.backward(); opt.step()
    return local

global_model=CaTDeLNet().to(DEVICE)
# initialize global parameters from centrally trained model for a warm-start experiment
global_model.load_state_dict(model.state_dict())
centers=sorted(df.center.astype(str).unique())
federated_log=[]
for rnd in range(FEDERATED_ROUNDS):
    client_states=[]; counts=[]; confidences=[]; ages=[]; client_protos=[]
    for ci,center in enumerate(centers[:NUM_CLIENTS]):
        cdf=df[df.center.astype(str)==center]
        if len(cdf)<2: continue
        cloader=DataLoader(ThyroidDataset(cdf),batch_size=BATCH_SIZE,shuffle=True,num_workers=0)
        local=train_local(global_model,cloader,LOCAL_EPOCHS)
        client_states.append(copy.deepcopy(local.state_dict())); counts.append(len(cdf))
        # Confidence proxy from local training objective; use bounded reproducible estimate.
        confidences.append(float(1/(1+np.exp(-min(len(cdf)/50,5)))))
        ages.append((rnd+ci)%3)
        _,cp=collect_embeddings(local,DataLoader(ThyroidDataset(cdf),batch_size=BATCH_SIZE))
        client_protos.append(cp.mean(0))
    if not client_states: print("No client data; skipping DPAA"); break
    state,agg_weights=dpaa_aggregate(global_model.state_dict(),client_states,counts,confidences,ages,client_protos)
    global_model.load_state_dict(state)
    federated_log.append({"round":rnd+1,"clients":len(client_states),"mean_client_samples":float(np.mean(counts)),
                          "weight_entropy":float(entropy(agg_weights+1e-12))})
    print(f"DPAA round {rnd+1}: clients={len(client_states)}, weights={np.round(agg_weights,3)}")
display(pd.DataFrame(federated_log))

## 9. Explainability: TDF maps and Grad-CAM-style saliency

TDF maps visualize absolute observed-versus-expected texture deviation. The saliency map below is a lightweight gradient-based class activation visualization (Grad-CAM-style), not a certified Grad-CAM++ implementation.

In [ ]:
# Cell 10 — TDF and gradient saliency visualization
def show_explanations(row_index=0):
    r=test_df.iloc[min(row_index,len(test_df)-1)]
    x=torch.tensor(preprocess_image(r.image_path)[None,None],dtype=torch.float32,device=DEVICE,requires_grad=True)
    model.eval()
    logits,sev,z,tdf,stats=model(x,return_aux=True)
    target=logits[:,1].sum()
    grad=torch.autograd.grad(target,x,retain_graph=False)[0].abs().mean(1)[0].detach().cpu().numpy()
    grad=cv2.resize(grad,(IMAGE_SIZE,IMAGE_SIZE))
    orig=x.detach().cpu().numpy()[0,0]
    tdfmap=tdf.detach().cpu().numpy()[0,0]
    fig,ax=plt.subplots(1,3,figsize=(12,4))
    ax[0].imshow(orig,cmap="gray"); ax[0].set_title("Preprocessed ROI")
    ax[1].imshow(tdfmap,cmap="inferno"); ax[1].set_title("Texture deviation field")
    ax[2].imshow(orig,cmap="gray"); ax[2].imshow(grad,cmap="jet",alpha=.45); ax[2].set_title("Gradient saliency (approx.)")
    for a in ax: a.axis("off")
    plt.tight_layout(); plt.show()
show_explanations(0)

## 10. Robustness and TDF stability

Evaluate probability retention under Gaussian noise, contrast change, intensity shift, translation, and resolution degradation. These tests use the held-out samples and report observed values; they do not replace external-center validation.

In [ ]:
# Cell 11 — Robustness stress tests
def perturb_image(x, kind):
    arr=np.asarray(x,dtype=np.float32).copy()
    if kind=="gaussian_noise": arr += np.random.normal(0,0.08,arr.shape)
    elif kind=="reduced_contrast": arr=0.55*arr
    elif kind=="intensity_shift": arr=arr+0.12
    elif kind=="translation": arr=np.roll(arr,shift=(3,-3),axis=(0,1))
    elif kind=="resolution_degradation":
        small=cv2.resize(arr,(IMAGE_SIZE//2,IMAGE_SIZE//2),interpolation=cv2.INTER_AREA)
        arr=cv2.resize(small,(IMAGE_SIZE,IMAGE_SIZE),interpolation=cv2.INTER_LINEAR)
    elif kind=="speckle": arr=arr*(1+np.random.normal(0,0.08,arr.shape))
    return np.clip(arr,-1,1).astype(np.float32)

@torch.no_grad()
def probability_for_array(arr):
    x=torch.tensor(arr[None,None],dtype=torch.float32,device=DEVICE)
    return float(torch.softmax(model(x),dim=1)[0,1].cpu())
robust_rows=[]
for kind in ["gaussian_noise","reduced_contrast","intensity_shift","translation","resolution_degradation","speckle"]:
    base=[]; pert=[]; tdf_sims=[]
    for _,r in test_df.iterrows():
        arr=preprocess_image(r.image_path)
        base.append(probability_for_array(arr))
        altered=perturb_image(arr,kind); pert.append(probability_for_array(altered))
        with torch.no_grad():
            xx=torch.tensor(arr[None,None],dtype=torch.float32,device=DEVICE)
            xp=torch.tensor(altered[None,None],dtype=torch.float32,device=DEVICE)
            t1=model(xx,return_aux=True)[3].flatten().cpu().numpy()
            t2=model(xp,return_aux=True)[3].flatten().cpu().numpy()
        tdf_sims.append(float(np.dot(t1,t2)/(np.linalg.norm(t1)*np.linalg.norm(t2)+1e-8)))
    retention=1-np.mean(np.abs(np.array(base)-np.array(pert)))
    robust_rows.append({"Perturbation":kind,"Probability retention (1-MAE)":retention,
                        "Mean TDF cosine similarity":np.mean(tdf_sims)})
robust_df=pd.DataFrame(robust_rows)
display(robust_df)

## 11. Calibration, bootstrap confidence intervals, and statistical tests

This section calculates ECE, reliability curve, bootstrap confidence intervals for key classification metrics, McNemar's paired error test, and a DeLong-style AUC comparison is intentionally omitted unless a suitable statistical package is installed. A p-value is not evidence of clinical usefulness by itself.

In [ ]:
# Cell 12 — Calibration, bootstrap 95% CI, McNemar test
def expected_calibration_error(y,p,n_bins=10):
    edges=np.linspace(0,1,n_bins+1); ece=0.0
    pred=(p>=0.5).astype(int)
    for lo,hi in zip(edges[:-1],edges[1:]):
        mask=(p>=lo)&(p<(hi if hi<1 else hi+1e-9))
        if mask.any():
            ece += mask.mean()*abs(float(pred[mask].mean())-float(y[mask].mean()))
    return ece

ece=expected_calibration_error(y_test,p_final)
frac_pos,mean_pred=calibration_curve(y_test,p_final,n_bins=min(10,max(2,len(y_test)//5)),strategy="uniform")
plt.figure(figsize=(5,5)); plt.plot(mean_pred,frac_pos,marker="o",label="CaTDeL-Net"); plt.plot([0,1],[0,1],"--",label="Perfect calibration")
plt.xlabel("Mean predicted probability"); plt.ylabel("Observed positive fraction"); plt.title(f"Reliability diagram (ECE={ece:.3f})"); plt.legend(); plt.grid(alpha=.3); plt.show()

def bootstrap_ci(y,p,n_boot=500,seed=SEED):
    rng=np.random.default_rng(seed); vals={k:[] for k in ["Accuracy","Sensitivity/Recall","Specificity","F1","MCC","ROC-AUC","PR-AUC"]}
    y=np.asarray(y); p=np.asarray(p)
    for _ in range(n_boot):
        ids=rng.integers(0,len(y),len(y)); yy=y[ids]; pp=p[ids]
        if len(np.unique(yy))<2: continue
        mm=classification_metrics(yy,pp)
        for k in vals: vals[k].append(mm[k])
    return pd.DataFrame([{"Metric":k,"Mean":np.nanmean(v),"CI_lower":np.nanpercentile(v,2.5),
                          "CI_upper":np.nanpercentile(v,97.5)} for k,v in vals.items()])
bootstrap_df=bootstrap_ci(y_test,p_final)
display(bootstrap_df)
print("ECE:",round(ece,4))

try:
    from statsmodels.stats.contingency_tables import mcnemar
    # Compare base neural classifier vs prototype-enriched output on same test examples.
    base_pred=(p_test>=0.5).astype(int)
    table=np.zeros((2,2),dtype=int)
    for a,b in zip(y_test!=base_pred,y_test!=y_pred): table[int(a),int(b)]+=1
    mc=mcnemar(table,exact=False,correction=True)
    print("McNemar table [[both correct, base only wrong?], ...]:\n",table)
    print("McNemar statistic:",mc.statistic,"p-value:",mc.pvalue)
except Exception as e:
    print("McNemar test skipped:",e)

## 12. Segmentation metrics (when masks exist)

Dice, IoU, sensitivity, specificity, and boundary distance are computed only for images with reference masks. If the segmentation model was not trained, no segmentation performance is reported.

In [ ]:
# Cell 13 — Optional segmentation evaluation
def dice_iou(pred,gt):
    pred=pred.astype(bool); gt=gt.astype(bool)
    inter=(pred&gt).sum(); union=(pred|gt).sum()
    dice=2*inter/(pred.sum()+gt.sum()+1e-8)
    iou=inter/(union+1e-8)
    return dice,iou

seg_results=[]
if segmentation_trained:
    segmenter.eval()
    for _,r in test_df.iterrows():
        mp=r.get("mask_path",None)
        if mp is None or pd.isna(mp) or not Path(str(mp)).exists(): continue
        x=torch.tensor(preprocess_image(r.image_path)[None,None],dtype=torch.float32,device=DEVICE)
        gt=cv2.imread(str(mp),cv2.IMREAD_GRAYSCALE)
        gt=cv2.resize(gt,(IMAGE_SIZE,IMAGE_SIZE),interpolation=cv2.INTER_NEAREST)>0
        with torch.no_grad(): pred=(torch.sigmoid(segmenter(x))[0,0].cpu().numpy()>=0.5)
        d,i=dice_iou(pred,gt); seg_results.append({"Dice":d,"IoU":i})
if seg_results:
    seg_df=pd.DataFrame(seg_results); display(seg_df.describe().T)
else:
    print("Segmentation metrics unavailable: no trained model/reference masks for the held-out set.")

## 13. TDF localization consistency, concentration, ablations, and resource reporting

TDLC/DCI require a meaningful reference lesion mask; without it, the code labels those measurements unavailable instead of fabricating scores. The ablation table is an experiment plan; run each variant with multiple seeds for publication-quality comparisons.

In [ ]:
# Cell 14 — TDF diagnostics and ablation configuration
def tdf_statistics(tdf, mask=None):
    a=np.abs(tdf).astype(np.float32)
    mean=float(a.mean()); variance=float(a.var())
    hist,_=np.histogram(a,bins=32,density=True); hist=hist[hist>0]
    ent=float(-(hist*np.log(hist+1e-12)).sum())
    top=max(1,a.size//10); concentration=float(np.partition(a.ravel(),-top)[-top:].mean()/(a.mean()+1e-8))
    out={"TDF_mean_abs":mean,"TDF_variance":variance,"TDF_entropy":ent,"TDF_spatial_concentration":concentration}
    if mask is not None and np.asarray(mask).sum()>0:
        mask=np.asarray(mask)>0
        out["Lesion_deviation_concentration"]=float(a[mask].mean()/(a[~mask].mean()+1e-8))
        high=a>=np.percentile(a,90)
        out["TDLC_top10_overlap"]=float((high&mask).sum()/(high.sum()+1e-8))
    else:
        out["Lesion_deviation_concentration"]=np.nan
        out["TDLC_top10_overlap"]=np.nan
    return out

tdf_diag=[]
for i in range(min(20,len(test_df))):
    r=test_df.iloc[i]
    x=torch.tensor(preprocess_image(r.image_path)[None,None],dtype=torch.float32,device=DEVICE)
    with torch.no_grad(): tdf=model(x,return_aux=True)[3][0,0].cpu().numpy()
    mask=None; mp=r.get("mask_path",None)
    if mp is not None and pd.notna(mp) and Path(str(mp)).exists():
        mask=cv2.imread(str(mp),cv2.IMREAD_GRAYSCALE)
        mask=cv2.resize(mask,(IMAGE_SIZE,IMAGE_SIZE),interpolation=cv2.INTER_NEAREST)>0
    tdf_diag.append(tdf_statistics(tdf,mask))
tdf_diag_df=pd.DataFrame(tdf_diag); display(tdf_diag_df.describe().T)

ablation_plan=pd.DataFrame([
 {"Variant":"A1 Baseline","Context Encoder":False,"Expected Texture":False,"TDF":False,"TDF Statistics":False},
 {"Variant":"A2","Context Encoder":True,"Expected Texture":False,"TDF":False,"TDF Statistics":False},
 {"Variant":"A3","Context Encoder":True,"Expected Texture":True,"TDF":True,"TDF Statistics":False},
 {"Variant":"A4 Proposed","Context Encoder":True,"Expected Texture":True,"TDF":True,"TDF Statistics":True},
 {"Variant":"B1 Local Model","TDF":True,"FedAvg":False,"DPAA":False},
 {"Variant":"B2 TDF + FedAvg","TDF":True,"FedAvg":True,"DPAA":False},
 {"Variant":"B3 Proposed TDF + DPAA","TDF":True,"FedAvg":False,"DPAA":True}
])
display(ablation_plan)
# Parameter count / model size (actual for this instantiated model)
n_params=sum(p.numel() for p in model.parameters())
model_size_mb=sum(p.numel()*p.element_size() for p in model.parameters())/(1024**2)
print(f"Parameters: {n_params:,} | parameter storage: {model_size_mb:.3f} MB")

## 14. Save artifacts

Exports held-out metrics, robustness, bootstrap intervals, DPAA round logs, TDF diagnostics, ablation configuration, and model weights. A ZIP archive is created in `OUTPUT_DIR`. Always retain the dataset provenance and evaluation split details alongside any reported results.

In [ ]:
# Cell 15 — Export results to CSV/Excel/ZIP
metrics_df.to_csv(OUTPUT_DIR/"classification_metrics.csv",index=False)
robust_df.to_csv(OUTPUT_DIR/"robustness_results.csv",index=False)
bootstrap_df.to_csv(OUTPUT_DIR/"bootstrap_confidence_intervals.csv",index=False)
pd.DataFrame(federated_log).to_csv(OUTPUT_DIR/"dpaa_round_log.csv",index=False)
tdf_diag_df.to_csv(OUTPUT_DIR/"tdf_diagnostics.csv",index=False)
ablation_plan.to_csv(OUTPUT_DIR/"ablation_plan.csv",index=False)
pd.DataFrame([{"data_mode":DATA_MODE,"n_rows_after_qc":len(df),"train_rows":len(train_df),
               "test_rows":len(test_df),"segmentation_trained":segmentation_trained,
               "parameters":n_params,"model_size_MB":model_size_mb,"ECE":ece}]).to_csv(OUTPUT_DIR/"run_summary.csv",index=False)
torch.save(model.state_dict(),OUTPUT_DIR/"catdel_model_state.pt")
torch.save(global_model.state_dict(),OUTPUT_DIR/"dpaa_global_model_state.pt")
with pd.ExcelWriter(OUTPUT_DIR/"CaTDeL_Results.xlsx",engine="openpyxl") as writer:
    metrics_df.to_excel(writer,sheet_name="Classification",index=False)
    robust_df.to_excel(writer,sheet_name="Robustness",index=False)
    bootstrap_df.to_excel(writer,sheet_name="Bootstrap_CI",index=False)
    pd.DataFrame(federated_log).to_excel(writer,sheet_name="DPAA",index=False)
    tdf_diag_df.to_excel(writer,sheet_name="TDF_Diagnostics",index=False)
    ablation_plan.to_excel(writer,sheet_name="Ablation_Plan",index=False)
    df.drop(columns=["severity_id"],errors="ignore").to_excel(writer,sheet_name="Manifest_QC",index=False)
zip_path=OUTPUT_DIR/"CaTDeL_Outputs.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as zf:
    for p in OUTPUT_DIR.iterdir():
        if p.is_file() and p.name != zip_path.name:
            zf.write(p,arcname=p.name)
print("Saved output folder:",OUTPUT_DIR)
print("Excel:",OUTPUT_DIR/"CaTDeL_Results.xlsx")
print("ZIP:",zip_path)
print("Notebook complete. If synthetic smoke-test mode was used, replace it with verified real data before scientific reporting.")